## 1. Chuẩn bị Google Colab

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## Dataset chuẩn 5 nhãn

Upload `posture_dataset_standard_5class.zip`. Cả bốn notebook dùng cùng train/val/test; không tự chia lại ảnh.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

ZIP_PATH = Path("/content/posture_dataset_standard_5class.zip")
DATA_DIR = Path("/content/posture_dataset_standard_5class")
CLASS_NAMES = ["leaning_backward", "leaning_forward", "leaning_left", "leaning_right", "upright"]

if not DATA_DIR.is_dir():
    if not ZIP_PATH.is_file():
        print("Chọn posture_dataset_standard_5class.zip để upload lên Colab")
        uploaded = files.upload()
        assert ZIP_PATH.name in uploaded, f"Cần upload đúng file {ZIP_PATH.name}"
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall("/content")

TRAIN_DIR = str(DATA_DIR / "train")
VAL_DIR = str(DATA_DIR / "val")
TEST_DIR = str(DATA_DIR / "test")
for split_dir in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    assert Path(split_dir).is_dir(), f"Thiếu split: {split_dir}"
    assert sorted(p.name for p in Path(split_dir).iterdir() if p.is_dir()) == CLASS_NAMES, f"Sai 5 nhãn trong {split_dir}"
print("Dataset chuẩn:", DATA_DIR)
print("Thứ tự nhãn:", CLASS_NAMES)


## 7. Load dữ liệu huấn luyện

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES, label_mode="int", shuffle=True, seed=SEED
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES, label_mode="int", shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_names=CLASS_NAMES, label_mode="int", shuffle=False
)
class_names = list(CLASS_NAMES)
assert train_ds.class_names == val_ds.class_names == test_ds.class_names == class_names
assert len(class_names) == 5
print("Classes:", class_names)


## 8. Kiểm tra dữ liệu trước khi huấn luyện mô hình

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize = (9, 9))
for images, labels in train_ds.take(1):
    n = min(9, len(images))
    for i in range(n):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

## 9. Prefetch dữ liệu

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 10. Học chuyển đổi (Transfer Learning) sử dụng MobileNetV2v

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False
inputs = keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(
    len(class_names),
    activation="softmax"
)(x)
model = keras.Model(inputs, outputs)
model.summary()

## 11. Compile mô hình

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 12. Huấn luyện mô hình

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

## 13. Training curves và Overfitting

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 14. Đánh giá trên tập dữ liệu test

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

## 15. Vẽ confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
y_true = []
y_pred = []

for images, labels in test_ds:
    y_true.extend(labels.numpy())
    predictions = model.predict(images)
    y_pred.extend(tf.argmax(predictions, axis=1).numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm)

ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 16. Dự đoán

In [ ]:
from google.colab import files
uploaded_image = files.upload()
image_filename = list(uploaded_image.keys())[0]
print("Uploaded:", image_filename)

In [ ]:
img = tf.keras.utils.load_img(
    image_filename,
    target_size=IMG_SIZE
)

img_array = tf.keras.utils.img_to_array(img)
batch = tf.expand_dims(img_array, axis=0)

pred = model.predict(batch, verbose=0)[0]
pred_index = int(np.argmax(pred))

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.title(f"{class_names[pred_index]} | score={pred[pred_index]:.3f}")
plt.show()

print("Prediction:", class_names[pred_index])
print("Model score:", float(pred[pred_index]))
print("\nScores by class:")

for name, score in zip(class_names, pred):
    print(f"- {name}: {float(score):.4f}")

## 17. Lưu lại mô hình

In [ ]:
MODEL_PATH = "/content/image_classifier_v1.keras"
model.save(MODEL_PATH)
print("Saved:", MODEL_PATH)

In [ ]:
# =========================================================
# EXPORT KERAS -> SAVEDMODEL -> ZIP -> DOWNLOAD
# =========================================================

import os
import shutil
import tensorflow as tf
from google.colab import files

KERAS_MODEL_PATH = "/content/image_classifier_v1.keras"
SAVED_MODEL_PATH = "/content/posture_saved_model"
ZIP_PATH = "/content/posture_saved_model.zip"

# 1. Xóa output cũ nếu có
if os.path.exists(SAVED_MODEL_PATH):
    shutil.rmtree(SAVED_MODEL_PATH)

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 2. Load model Keras
print("1. Loading Keras model...")

model = tf.keras.models.load_model(
    KERAS_MODEL_PATH,
    compile=False
)

print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)

# 3. Export sang SavedModel
print("\n2. Exporting SavedModel...")

model.export(SAVED_MODEL_PATH)

# 4. Kiểm tra file đã export
print("\n3. SavedModel files:")

for root, dirs, filenames in os.walk(SAVED_MODEL_PATH):
    level = root.replace(SAVED_MODEL_PATH, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for filename in filenames:
        full_path = os.path.join(root, filename)
        size_mb = os.path.getsize(full_path) / 1024 / 1024

        print(
            f"{indent}  {filename} "
            f"({size_mb:.2f} MB)"
        )

# 5. Zip toàn bộ SavedModel
print("\n4. Creating ZIP...")

import json
with open(os.path.join(SAVED_MODEL_PATH, "class_names.json"), "w") as stream:
    json.dump(class_names, stream)

shutil.make_archive(
    "/content/posture_saved_model",
    "zip",
    SAVED_MODEL_PATH
)

print("\n✅ Done")
print("SavedModel:", SAVED_MODEL_PATH)
print("ZIP:", ZIP_PATH)

# 6. Download về máy
files.download(ZIP_PATH)